# EDA — Physionomie_Transactions_MERGED.csv

In [1]:
import pandas as pd
import numpy as np

df_physio = pd.read_csv('D:/rouge_file/DATA_BULLETINS_CSV/Physionomie_Transactions_MERGED.csv')


In [2]:
print(f'Shape : {df_physio.shape}')


Shape : (435705, 6)


In [3]:
df_physio.head(5)

,Type,Instruments,Cours,Quantité,Volume non doublé MAD,Nombre de contrats
0,Type,MARCHE CENTRAL,NaN,670 973,"39 985 775,19",1 131
1,Non spécifié,ACTIONS,NaN,670 955,"39 985 729,47",1 128
2,Non spécifié,AFMA,"1 296,00",1,"1 296,00",1
3,Non spécifié,AFRIC INDUSTRIES SA,"354,00",14,"4 956,00",3
4,Non spécifié,ALLIANCES,"64,20",1 000,"64 200,00",1


## 1. Qualite des donnees

In [4]:
print('Valeurs manquantes :')
print(df_physio.isnull().sum())


Valeurs manquantes :
Type                         0
Instruments                  0
Cours                    17259
Quantité                    13
Volume non doublé MAD       10
Nombre de contrats          20
dtype: int64


In [5]:
print(f'\nDoublons         : {df_physio.duplicated().sum()}')


Doublons         : 23327


In [6]:
print(f'Lignes parasites : {(df_physio["Type"] == "Type").sum()}')


Lignes parasites : 699


In [7]:
print('\nValeurs colonne Type :')
print(df_physio['Type'].value_counts())


Valeurs colonne Type :
Type
Non spécifié      434996
Type                 699
non doublé MAD         9
ans doublé MAD         1
Name: count, dtype: int64


## 2. Nettoyage

### Suppression parasites (Type=Type, lignes residuelles OCR) + doublons


In [8]:
parasites = ['Type', 'non doublé MAD', 'ans doublé MAD']
df_physio = df_physio[~df_physio['Type'].isin(parasites)].drop_duplicates().copy()

def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(' ','').str.replace('\xa0','')
         .str.replace(',','.').str.strip(),
        errors='coerce'
    )

df_physio['Cours']    = to_num(df_physio['Cours'])
df_physio['Quantite'] = to_num(df_physio['Quantité'])
df_physio['Volume']   = to_num(df_physio['Volume non doublé MAD'])
df_physio['Contrats'] = to_num(df_physio['Nombre de contrats'])



### Separer lignes agregats (MARCHE CENTRAL, ACTIONS, TOTAL...) des lignes titre


In [9]:
agregats = ['MARCHE CENTRAL','ACTIONS','TOTAL GENERAL','MARCHE DE BLOCS',
            'APPORTS DE TITRES','AUGMENTATION DE CAPITAL EN NUMERAIRE',
            'OBLIGATIONS','TRANSFERTS']
df_titres   = df_physio[~df_physio['Instruments'].isin(agregats)].copy()
df_agregats = df_physio[df_physio['Instruments'].isin(agregats)].copy()

print(f'Shape total apres nettoyage : {df_physio.shape}')
print(f'Lignes titres individuels   : {len(df_titres):,}')
print(f'Lignes agregats marche      : {len(df_agregats):,}')
df_titres.head(5)

Shape total apres nettoyage : (411686, 9)
Lignes titres individuels   : 407,992
Lignes agregats marche      : 3,694


,Type,Instruments,Cours,Quantité,Volume non doublé MAD,Nombre de contrats,Quantite,Volume,Contrats
2,Non spécifié,AFMA,1296.0,1,"1 296,00",1,1.0,1296.0,1.0
3,Non spécifié,AFRIC INDUSTRIES SA,354.0,14,"4 956,00",3,14.0,4956.0,3.0
4,Non spécifié,ALLIANCES,64.2,1 000,"64 200,00",1,1000.0,64200.0,1.0
5,Non spécifié,ALLIANCES,65.0,2 916,"189 540,00",4,2916.0,189540.0,4.0
6,Non spécifié,ALLIANCES,65.1,200,"13 020,00",1,200.0,13020.0,1.0


## 3. Statistiques descriptives

In [10]:
df_titres[['Cours','Quantite','Volume','Contrats']].describe().round(2)

,Cours,Quantite,Volume,Contrats
count,407468.00,407992.00,4.079920e+05,4.079920e+05
mean,611.20,3463.30,7.902102e+05,1.721030e+03
std,1081.58,103557.95,2.940132e+07,5.006275e+05
min,0.39,1.00,1.000000e+00,1.000000e+00
25%,92.55,18.00,5.529300e+03,1.000000e+00
50%,298.15,100.00,2.608525e+04,2.000000e+00
75%,679.00,510.00,1.211914e+05,5.000000e+00
max,105892.85,28392646.00,1.216892e+10,2.878477e+08


## 4. Analyse par Instrument

In [11]:
print(f'Nombre de titres distincts : {df_titres["Instruments"].nunique()}')
print()


Nombre de titres distincts : 240



In [12]:
print('Top 15 titres par volume total echange (MAD) :')
top_vol = df_titres.groupby('Instruments')['Volume'].sum().sort_values(ascending=False).head(15)
print(top_vol.apply(lambda x: f'{x:,.0f}').to_string())

Top 15 titres par volume total echange (MAD) :
Instruments
BCP                     28,091,394,174
CIMENTS DU MAROC        18,664,105,436
ATTUARIWAFA BANK        18,581,831,662
TGCC S.A                17,685,983,196
ATTIJARIWAFA BANK       13,801,252,782
ITISSALAT AL-MAGHRIB    13,769,529,345
SODEP-Marsa Maroc       13,394,105,918
AKDITAL                 13,335,115,092
BANK OF AFRICA          12,267,261,303
COSUMAR                 11,635,939,666
SGTM S.A                10,894,325,857
DOUJA PROM ADDOHA       10,714,411,303
MANAGEM                 10,641,692,494
ALLIANCES               10,073,216,162
INTRODUCTIONS            7,778,704,254


In [13]:
print('Top 15 titres par nombre de contrats :')
top_ctrats = df_titres.groupby('Instruments')['Contrats'].sum().sort_values(ascending=False).head(15)
print(top_ctrats.apply(lambda x: f'{x:,.0f}').to_string())
print()

Top 15 titres par nombre de contrats :
Instruments
RISMA                            289,011,990
TAQA MOROCCO                     111,566,077
BANK OF AFRICA                    66,534,188
MUTANDIS SCA                      43,075,886
ATTUARIWAFA BANK                  36,312,029
TGCC S.A                          24,450,090
BCP                               19,869,011
SODEP-Marsa Maroc                 14,383,606
WAFA ASSURANCE                    14,255,645
DOUJA PROM ADDOHA                 12,050,824
LafargeHolcim Maroc               10,865,870
CIMENTS DU MAROC                   9,548,969
SONASID                            7,391,033
RESIDENCES DAR SAADA               4,617,127
TOTALENERGIES MARKETING MAROC      4,392,396



In [14]:
print('Top 15 titres par quantite echangee :')
top_qte = df_titres.groupby('Instruments')['Quantite'].sum().sort_values(ascending=False).head(15)
print(top_qte.apply(lambda x: f'{x:,.0f}').to_string())

Top 15 titres par quantite echangee :
Instruments
DOUJA PROM ADDOHA       396,579,575
ITISSALAT AL-MAGHRIB    134,236,857
BCP                      97,523,511
RESIDENCES DAR SAADA     61,959,214
BANK OF AFRICA           58,569,774
COSUMAR                  56,772,121
DROITS                   51,130,126
ATTUARIWAFA BANK         34,111,945
ALLIANCES                33,465,112
DELTA HOLDING            32,964,597
CFG BANK                 32,748,200
TGCC S.A                 31,264,461
INTRODUCTIONS            28,873,189
DS CIH 3/23 2025         25,890,746
STOKVIS NORD AFRIQUE     25,692,394


## 5. Analyse des Cours

In [15]:
print('Stats cours par titre (moyenne, min, max) :')
stats_cours = df_titres.groupby('Instruments')['Cours'].agg(
    Cours_Moyen = 'mean',
    Cours_Min   = 'min',
    Cours_Max   = 'max',
    Nb_seances  = 'count'
).round(2).sort_values('Cours_Moyen', ascending=False)


Stats cours par titre (moyenne, min, max) :


In [16]:
print('Top 10 cours les plus eleves :')
print(stats_cours.head(10).to_string())
print()

Top 10 cours les plus eleves :
                              Cours_Moyen  Cours_Min  Cours_Max  Nb_seances
Instruments                                                                
29JUN2015 4.77% 10A 100K SOG    105374.84  105374.84  105374.84           1
12OCT2016 4,43% 10A 100K CAM    105359.51  105359.51  105359.51           1
27NOV2015 4,80% 10A 100K CAM    105081.97  105081.97  105081.97           1
27NOV2015 4.80% 10A 100K CAM    104483.93  104483.93  104483.93           1
11OCT2017 4,22% 10A 100K CAM    104216.70  102540.55  105892.85           2
12OCT2016 4.11% 7A 100K CAM     103753.37  103753.37  103753.37           1
22DEC2015 4.52% 10A 100K ATW    102894.70  102894.70  102894.70           1
22DEC2014 4,75% 10A 100K ATW    101891.86  101891.86  101891.86           1
26JUN2016 3,74% 10A 100K ATW    101231.58  101231.58  101231.58           1
28JUN2016 3.74% 10A 100K BOA    101008.74  100816.55  101200.93           2



In [17]:
print('Top 10 cours les plus bas :')
print(stats_cours.tail(10).to_string())

Top 10 cours les plus bas :
                                   Cours_Moyen  Cours_Min  Cours_Max  Nb_seances
Instruments                                                                     
APPOATS DE TITRES                          NaN        NaN        NaN           0
APPOIRTS DE TITRES                         NaN        NaN        NaN           0
AUGMENTATION PAR APPORT EN NATURE          NaN        NaN        NaN           0
DROITS                                     NaN        NaN        NaN           0
INTRODUCTIONS                              NaN        NaN        NaN           0
MARCHE DE BLOCK                            NaN        NaN        NaN           0
OFFRE PUBLIQUE D'ACHAT                     NaN        NaN        NaN           0
OFFRE PUBLIQUE DE RETRAIT                  NaN        NaN        NaN           0
OFFRE PUBLIQUE DE VENTE                    NaN        NaN        NaN           0
TOTAL GÉNÉRAL                              NaN        NaN        NaN           0


## 6. Analyse du Volume Global du Marche

In [18]:
print('Volume global par segment de marche :')
print(df_agregats.groupby('Instruments')['Volume'].sum()
      .sort_values(ascending=False)
      .apply(lambda x: f'{x:,.0f} MAD').to_string())
print()


Volume global par segment de marche :
Instruments
MARCHE CENTRAL                          360,001,328,430 MAD
ACTIONS                                 334,844,772,784 MAD
TOTAL GENERAL                           313,304,239,213 MAD
MARCHE DE BLOCS                          64,296,659,023 MAD
APPORTS DE TITRES                        13,924,623,597 MAD
AUGMENTATION DE CAPITAL EN NUMERAIRE      8,893,648,440 MAD
TRANSFERTS                                2,125,222,171 MAD
OBLIGATIONS                                 480,160,069 MAD



### Part de chaque titre dans le volume total


In [19]:
volume_total = df_titres['Volume'].sum()
part_titres  = (df_titres.groupby('Instruments')['Volume'].sum() / volume_total * 100).round(2)
print('Top 10 titres — part dans le volume total :')
print(part_titres.sort_values(ascending=False).head(10).to_string())

Top 10 titres — part dans le volume total :
Instruments
BCP                     8.71
CIMENTS DU MAROC        5.79
ATTUARIWAFA BANK        5.76
TGCC S.A                5.49
ATTIJARIWAFA BANK       4.28
ITISSALAT AL-MAGHRIB    4.27
SODEP-Marsa Maroc       4.15
AKDITAL                 4.14
BANK OF AFRICA          3.80
COSUMAR                 3.61


## 7. Outliers

In [20]:
for col in ['Cours','Quantite','Volume','Contrats']:
    Q1, Q3 = df_titres[col].quantile(0.25), df_titres[col].quantile(0.75)
    IQR    = Q3 - Q1
    out    = df_titres[(df_titres[col] < Q1-1.5*IQR) | (df_titres[col] > Q3+1.5*IQR)]
    print(f'{col:12s} : {len(out):,} outliers ({len(out)/len(df_titres)*100:.1f}%)')

Cours        : 43,894 outliers (10.8%)
Quantite     : 64,993 outliers (15.9%)
Volume       : 62,498 outliers (15.3%)
Contrats     : 42,236 outliers (10.4%)


## 8. Correlation

In [21]:
df_titres[['Cours','Quantite','Volume','Contrats']].corr().round(4)

,Cours,Quantite,Volume,Contrats
Cours,1.0000,-0.0128,0.0106,-0.0121
Quantite,-0.0128,1.0000,0.5364,-0.0001
Volume,0.0106,0.5364,1.0000,-0.0000
Contrats,-0.0121,-0.0001,-0.0000,1.0000


## 9. Export

In [22]:
df_physio.to_csv('Physionomie_CLEAN.csv', index=False, encoding='utf-8')
print(f'Exporte : Physionomie_CLEAN.csv — {len(df_physio):,} lignes')

Exporte : Physionomie_CLEAN.csv — 411,686 lignes
